# S03 Seoul bike sharing: a first look
Session 3 EDA: data understanding. Demo dataset: Seoul Bike Sharing Demand (UCI, CC BY 4.0), the clean reference version. Goal: know what is in the file **before** cleaning, plotting or modelling.

## 1. Load seoul_reference.csv: 8,760 rows, one per hour, Dec 2017–Nov 2018

In [1]:
import pandas as pd

df = pd.read_csv("../data/seoul_reference.csv")
len(df)

8760

## 2. shape, head, dtypes: what does one row mean?

In [2]:
df.shape

(8760, 14)

In [3]:
df.head()

,timestamp,rented_bike_count,temp_c,humidity_pct,wind_speed_m_s,visibility_m,dew_point_c,solar_radiation_mj_m2,rainfall_mm,snowfall_cm,season,holiday,functioning_day,split
0,2017-12-01 00:00,254,-5.2,37,2.2,20000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train
1,2017-12-01 01:00,204,-5.5,38,0.8,20000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train
2,2017-12-01 02:00,173,-6.0,39,1.0,20000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,train
3,2017-12-01 03:00,107,-6.2,40,0.9,20000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train
4,2017-12-01 04:00,78,-6.0,36,2.3,20000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes,train


In [4]:
df.dtypes

timestamp                    str
rented_bike_count          int64
temp_c                   float64
humidity_pct               int64
wind_speed_m_s           float64
visibility_m               int64
dew_point_c              float64
solar_radiation_mj_m2    float64
rainfall_mm              float64
snowfall_cm              float64
season                       str
holiday                      str
functioning_day              str
split                        str
dtype: object

**One row = one hour in Seoul**: how many bikes were rented in that hour, plus the weather at that time. `timestamp`, `season`, `holiday`, `functioning_day` and `split` are text (`object`, or `str` in pandas 3); everything else is numeric.

## 3. describe: which ranges look plausible?

In [5]:
df.describe().round(1).T

,count,mean,std,min,25%,50%,75%,max
rented_bike_count,8760.0,704.6,645.0,0.0,191.0,504.5,1065.2,3556.0
temp_c,8760.0,12.9,11.9,-17.8,3.5,13.7,22.5,39.4
humidity_pct,8760.0,58.2,20.4,0.0,42.0,57.0,74.0,98.0
wind_speed_m_s,8760.0,1.7,1.0,0.0,0.9,1.5,2.3,7.4
visibility_m,8760.0,14368.3,6083.0,270.0,9400.0,16980.0,20000.0,20000.0
dew_point_c,8760.0,4.1,13.1,-30.6,-4.7,5.1,14.8,27.2
solar_radiation_mj_m2,8760.0,0.6,0.9,0.0,0.0,0.0,0.9,3.5
rainfall_mm,8760.0,0.1,1.1,0.0,0.0,0.0,0.0,35.0
snowfall_cm,8760.0,0.1,0.4,0.0,0.0,0.0,0.0,8.8


Read it with domain knowledge: temperatures between about −18 and 39 °C fit Seoul. Humidity has a minimum of **0 %**, which never happens outdoors. Visibility tops out at exactly 20,000 m in many rows (the sensor's maximum). And `rented_bike_count` has a minimum of **0**: suspicious? We look at both below.

## 4. value_counts on season, holiday, functioning day

In [6]:
for col in ["season", "holiday", "functioning_day"]:
    print(df[col].value_counts(), "\n")

season
Spring    2208
Summer    2208
Autumn    2184
Winter    2160
Name: count, dtype: int64 

holiday
No Holiday    8328
Holiday        432
Name: count, dtype: int64 

functioning_day
Yes    8465
No      295
Name: count, dtype: int64 



## 5. Parse timestamp and derive hour, weekday, month; meet split
The CSV stores the time as text. `pd.to_datetime` turns it into a real date-time; then we derive the calendar features we want.

In [7]:
ts = pd.to_datetime(df["timestamp"])
df["hour"] = ts.dt.hour
df["weekday"] = ts.dt.day_of_week   # 0 = Monday
df["month"] = ts.dt.month
df[["timestamp", "hour", "weekday", "month"]].head()

,timestamp,hour,weekday,month
0,2017-12-01 00:00,0,4,12
1,2017-12-01 01:00,1,4,12
2,2017-12-01 02:00,2,4,12
3,2017-12-01 03:00,3,4,12
4,2017-12-01 04:00,4,4,12


In [8]:
ts.min(), ts.max()

(Timestamp('2017-12-01 00:00:00'), Timestamp('2018-11-30 23:00:00'))

In [9]:
df["split"].value_counts()

split
train    6576
test     2184
Name: count, dtype: int64

In [10]:
df.groupby("split")["timestamp"].agg(["min", "max"])

,min,max
split,,
test,2018-09-01 00:00,2018-11-30 23:00
train,2017-12-01 00:00,2018-08-31 23:00


`split` is the prescribed train/test split: **train** = Dec 2017–Aug 2018, **test** = Sep–Nov 2018. We use it from Session 7 on.

## 6. Build the data dictionary as a small DataFrame
Meaning and expected range come from **us** (the column names and what we know about Seoul); type, observed range and missing share come from the data. Every "no" in *plausible?* is a finding.

In [11]:
meaning = {
    "timestamp": ("start of the hour (local time)", None, None),
    "rented_bike_count": ("bikes rented in that hour (target)", 0, 5000),
    "temp_c": ("air temperature in °C", -25, 40),
    "humidity_pct": ("relative humidity in %", 5, 100),
    "wind_speed_m_s": ("wind speed in m/s", 0, 20),
    "visibility_m": ("visibility in m", 0, 20000),
    "dew_point_c": ("dew point temperature in °C", -35, 30),
    "solar_radiation_mj_m2": ("solar radiation in MJ/m² per hour", 0, 4),
    "rainfall_mm": ("rain in mm per hour", 0, 50),
    "snowfall_cm": ("new snow in cm", 0, 10),
    "season": ("season of the year", None, None),
    "holiday": ("public holiday or not", None, None),
    "functioning_day": ("was the rental system operating?", None, None),
    "split": ("prescribed train/test split", None, None),
}
rows = []
for col, (text, lo, hi) in meaning.items():
    s = df[col]
    numeric = pd.api.types.is_numeric_dtype(s)
    observed = f"{s.min():g} to {s.max():g}" if numeric else f"{s.nunique()} values"
    expected = f"{lo} to {hi}" if lo is not None else "-"
    plausible = ("yes" if s.between(lo, hi).all() else "no") if lo is not None else "check values"
    rows.append({"column": col, "type": str(s.dtype), "meaning": text, "expected": expected,
                 "observed": observed, "missing %": round(s.isna().mean() * 100, 1),
                 "plausible?": plausible})
data_dictionary = pd.DataFrame(rows)
data_dictionary

,column,type,meaning,expected,observed,missing %,plausible?
0,timestamp,str,start of the hour (local time),-,8760 values,0.0,check values
1,rented_bike_count,int64,bikes rented in that hour (target),0 to 5000,0 to 3556,0.0,yes
2,temp_c,float64,air temperature in °C,-25 to 40,-17.8 to 39.4,0.0,yes
3,humidity_pct,int64,relative humidity in %,5 to 100,0 to 98,0.0,no
4,wind_speed_m_s,float64,wind speed in m/s,0 to 20,0 to 7.4,0.0,yes
5,visibility_m,int64,visibility in m,0 to 20000,270 to 20000,0.0,yes
6,dew_point_c,float64,dew point temperature in °C,-35 to 30,-30.6 to 27.2,0.0,yes
7,solar_radiation_mj_m2,float64,solar radiation in MJ/m² per hour,0 to 4,0 to 3.52,0.0,yes
8,rainfall_mm,float64,rain in mm per hour,0 to 50,0 to 35,0.0,yes
9,snowfall_cm,float64,new snow in cm,0 to 10,0 to 8.8,0.0,yes


One "no": humidity. Which rows?

In [12]:
df.loc[df["humidity_pct"] < 5, ["timestamp", "humidity_pct", "temp_c", "rainfall_mm"]]

,timestamp,humidity_pct,temp_c,rainfall_mm
4063,2018-05-19 07:00,0,11.4,0.0
4106,2018-05-21 02:00,0,13.9,0.0
4107,2018-05-21 03:00,0,13.0,0.0
4108,2018-05-21 04:00,0,12.4,0.0
4109,2018-05-21 05:00,0,11.9,0.0
4110,2018-05-21 06:00,0,11.4,0.0
4111,2018-05-21 07:00,0,12.3,0.0
4131,2018-05-22 03:00,0,15.5,0.0
4132,2018-05-22 04:00,0,15.6,0.0
4133,2018-05-22 05:00,0,15.7,0.0


17 hours on five days in May 2018 show exactly 0 %: almost certainly a sensor problem, not weather. A real data issue even in the "clean" file: note it for Session 5.

## Ask the room: some hours have 0 rentals. Error or real?

In [13]:
zero = df["rented_bike_count"] == 0
print("hours with 0 rentals:", zero.sum())
pd.crosstab(zero, df["functioning_day"], rownames=["0 rentals"])

hours with 0 rentals: 295


functioning_day,No,Yes
0 rentals,,
False,0,8465
True,295,0


All 0-rental hours are hours with `functioning_day = No`: the system **was not operating**. Real, not an error, but those hours tell a model nothing about demand. What to do with them is a decision for Session 5 (and it belongs in the cleaning log).